# Benchmark: ARIMA Statistical Baseline

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## Purpose

ARIMA serves as the canonical **univariate statistical baseline**.  
Because ARIMA cannot ingest multi-dimensional feature matrices, it operates  
exclusively on the **target time series** `energy_delta_k`.  
This structural limitation must be clearly stated in the paper:

> *"The ARIMA model was evaluated in a univariate setting owing to its  
> architectural constraint of accepting a single input series.  
> All other models were trained on the full multi-variate feature set."*

This distinction does **not** disadvantage the comparison — reviewers  
recognise ARIMA as a well-understood statistical floor, not a deep learning  
competitor.

### What this notebook does
1. Loads the frozen canonical splits for a given horizon.  
2. Reconstructs the raw `y` series (train → val → test in strict order).  
3. Fits an ARIMA model by grid-searching over `(p, d, q)` using AIC.  
4. Generates a rolling one-step-ahead forecast over the test window.  
5. Saves predictions, metrics, and diagnostic plots.


In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools   import adfuller
except ImportError:
    raise ImportError('Run:  pip install statsmodels')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config  import SPLITS_DIR, BENCHMARK_DIR, RESULTS_DIR, FORECAST_HORIZONS
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import compute_metrics, save_benchmark_results, build_comparison_table
from core.config import TARGET_LOG_TRANSFORM

sns.set(style='whitegrid')
ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)
print(f'Project root : {PROJECT_ROOT}')
print('Environment ready.')

In [ ]:
# ── Edit these lines only ───────────────────────────────────────────────────
HORIZONS_TO_RUN = [60, 300, 900, 1800]
SKIP_COMPLETED  = False

# ARIMA fits poorly to >50k rows and has O(n) complexity per step.
# These caps keep runtime to ~10-20 min per horizon.
MAX_ARIMA_TRAIN = 50_000    # rows used for AIC grid search + final fit
MAX_TEST_ROWS   = 5_000     # rolling forecast rows (most time-consuming part)
# ───────────────────────────────────────────────────────────────────────────

for _h in HORIZONS_TO_RUN:
    assert _h in FORECAST_HORIZONS
print(f'Horizons queued : {HORIZONS_TO_RUN}')
print(f'ARIMA train cap : {MAX_ARIMA_TRAIN:,}  |  test cap : {MAX_TEST_ROWS:,}')

---
## 1 · Stationarity Check

The Augmented Dickey-Fuller test confirms whether the target series  
is stationary.  ARIMA differencing order `d` is selected accordingly.

---
## 2 · (p, q) Grid Search via AIC

A small grid is searched to keep runtime tractable.  
A training subset is used; ARIMA scales poorly to millions of rows.

---
## 3 · Rolling One-Step-Ahead Forecast

The model is re-fitted on an expanding window as each test observation  
is consumed.  This is the most defensible evaluation strategy for ARIMA  
in a research context, as it avoids multi-step forecast accumulation error.

---
## 4 · Evaluation and Artefact Saving

---
## 5 · Diagnostic Plots

In [ ]:
all_results = {}
loop_start  = time.time()

for _loop_idx, HORIZON in enumerate(HORIZONS_TO_RUN):

    _elapsed = (time.time() - loop_start) / 3600
    print(f'\n' + '=' * 60)
    print(f'  Horizon {_loop_idx+1}/{len(HORIZONS_TO_RUN)} : '
          f'k = {HORIZON} s   {_elapsed:.2f} h elapsed')
    print('=' * 60)

    if SKIP_COMPLETED:
        _m_path = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}', 'arima', 'metrics.json')
        if os.path.isfile(_m_path):
            print('  [SKIP] ARIMA already evaluated for this horizon.')
            continue

    # ── Load splits (y arrays only — ARIMA is univariate) ───────────────────
    print('  Loading splits ...')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
        load_splits(SPLITS_DIR, HORIZON)
    print(f'  train={len(y_train):,}  val={len(y_val):,}  test={len(y_test):,}')

    # ── ADF stationarity test ────────────────────────────────────────────────
    print('  ADF stationarity test ...')
    _sample    = y_train[:20_000]
    _adf, _p, _, _, _crit, _ = adfuller(_sample, autolag='AIC')
    _is_stat   = _p < 0.05
    DIFF_ORDER = 0 if _is_stat else 1
    print(f'  ADF={_adf:.4f}  p={_p:.6f}  '
          f'({"stationary" if _is_stat else "non-stationary"})  d={DIFF_ORDER}')

    # ── (p, d, q) grid search by AIC ────────────────────────────────────────
    _y_fit   = np.concatenate([y_train, y_val])[:MAX_ARIMA_TRAIN]
    _best_aic   = np.inf
    _best_order = (1, DIFF_ORDER, 1)
    _aic_rows   = []

    print(f'  Grid searching ARIMA(p,{DIFF_ORDER},q) on {len(_y_fit):,} rows ...')
    for _p_ord in range(0, 4):
        for _q_ord in range(0, 4):
            try:
                _fit = ARIMA(_y_fit, order=(_p_ord, DIFF_ORDER, _q_ord)).fit()
                _aic_rows.append({'p': _p_ord, 'd': DIFF_ORDER,
                                  'q': _q_ord, 'AIC': _fit.aic})
                if _fit.aic < _best_aic:
                    _best_aic   = _fit.aic
                    _best_order = (_p_ord, DIFF_ORDER, _q_ord)
            except Exception:
                pass

    _aic_df = pd.DataFrame(_aic_rows).sort_values('AIC').reset_index(drop=True)
    print(f'  Best order : ARIMA{_best_order}  AIC={_best_aic:.2f}')
    print('  Top 3 by AIC:')
    print(_aic_df.head(3).to_string(index=False))

    # ── Rolling one-step-ahead forecast ────────────────────────────────────
    _n_test   = min(len(y_test), MAX_TEST_ROWS)
    _y_test_s = y_test[:_n_test]
    _history  = list(_y_fit)
    _preds    = []

    print(f'  Rolling forecast over {_n_test:,} test samples ...')
    for _t, _obs in enumerate(_y_test_s):
        try:
            _arima = ARIMA(_history, order=_best_order).fit()
            _fc    = float(_arima.forecast(steps=1)[0])
        except Exception:
            _fc = float(np.mean(_history[-100:]))
        _preds.append(_fc)
        _history.append(_obs)
        if (_t + 1) % 500 == 0:
            print(f'    ... {_t+1}/{_n_test}')

    _y_pred_arima = np.array(_preds, dtype=np.float32)
    print('  Rolling forecast complete.')

    # ── Save results ────────────────────────────────────────────────────────
    _m = save_benchmark_results(
        model_name='arima',
        horizon=HORIZON,
        y_true=_y_test_s,
        y_pred=_y_pred_arima,
        benchmark_root=BENCHMARK_DIR,
        log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={
            'order': list(_best_order),
            'best_aic': float(_best_aic),
            'training_size': int(MAX_ARIMA_TRAIN),
            'test_subset': int(_n_test),
            'note': 'Univariate model — uses target series only (log1p space)',
        },
    )
    print(f'  ARIMA{_best_order}  →  '
          f'MAE={_m["MAE"]:.6f}  RMSE={_m["RMSE"]:.6f}  R2={_m["R2"]:.4f} sMAPE={_m["sMAPE"]:.4F}')
    all_results[HORIZON] = _m

    # ── Diagnostic plots ────────────────────────────────────────────────────
    _pd = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}', 'arima', 'plots')
    os.makedirs(_pd, exist_ok=True)

    _n_show = min(300, _n_test)

    # Time-series overlay
    _fig, _ax = plt.subplots(figsize=(12, 4))
    _ax.plot(_y_test_s[:_n_show],      lw=1.2, label='Actual')
    _ax.plot(_y_pred_arima[:_n_show],  lw=1.2, linestyle='--', label='ARIMA')
    _ax.set_xlabel('Test sample index')
    _ax.set_ylabel(f'ΔE  k={HORIZON}s (kWh)')
    _ax.set_title(f'ARIMA{_best_order} — Actual vs Predicted  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'arima_overlay.png'), dpi=200)
    plt.close('all')

    # Residuals
    _fig, _ax = plt.subplots(figsize=(7, 4))
    sns.histplot(_y_test_s - _y_pred_arima, bins=50, kde=True,
                 ax=_ax, color='steelblue')
    _ax.axvline(0, color='red', lw=1.2, linestyle='--')
    _ax.set_xlabel('Residual (kWh)')
    _ax.set_title(f'ARIMA Residual Distribution  k={HORIZON}s')
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'arima_residuals.png'), dpi=200)
    plt.close('all')

    # AIC heatmap
    if len(_aic_df) >= 4:
        _pivot = _aic_df.pivot(index='p', columns='q', values='AIC')
        _fig, _ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(_pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=_ax)
        _ax.set_title(f'AIC Surface ARIMA(p,{DIFF_ORDER},q)  k={HORIZON}s')
        plt.tight_layout()
        plt.savefig(os.path.join(_pd, 'aic_heatmap.png'), dpi=200)
        plt.close('all')

    print(f'  Plots saved → {_pd}')

    del X_train, y_train, X_val, y_val, X_test, y_test, _y_pred_arima
    print(f'\n  DONE k={HORIZON}s')
    print('=' * 60)

_total_h = (time.time() - loop_start) / 3600
print(f'\nARIMA complete — {_total_h:.2f} h total')

In [ ]:
if not all_results:
    print('No results yet — run Cell 3 first.')
else:
    _rows = [{'Horizon_k_s': _h, 'Model': 'arima', **_m}
             for _h, _m in sorted(all_results.items())]
    _df = pd.DataFrame(_rows)
    print('ARIMA — all horizons')
    print('=' * 60)
    print(_df.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 60)
    _csv = os.path.join(ANALYSIS_DIR, 'arima_all_horizons.csv')
    _df.to_csv(_csv, index=False)
    print(f'Saved → {_csv}')